In [1]:
import faiss

In [2]:
import torch

In [3]:
import numpy as np

In [3]:
# pip install pymupdf
# pip install superkmeans
# pip install ollama

In [4]:
print("Torch version:", torch.__version__)
print("Built with CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.11.0+cu128
Built with CUDA: 12.8
CUDA available: True
GPU count: 1
GPU: NVIDIA GeForce GTX 1650


In [5]:
from sentence_transformers import SentenceTransformer

D:\Anaconda3\envs\financial_analysis\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Reading document by each page

- Ensure that we are getting the necessary metadata such as the file name, title, author, pages, page no. 
- This can help in searching based on metadata
- Can also aid in expanding the context of the retrieved chunks before passing to the LLM

In [7]:
import fitz
from langchain_core.documents import Document
from pathlib import Path

docs = []

for pdf_path in Path(r"D:\Github desktop\RAG\pdf files").glob("*.pdf"):
    print(pdf_path)
    pdf = fitz.open(pdf_path)
    pdf_metadata = pdf.metadata
    # print(pdf_metadata)

    for page_num, page in enumerate(pdf):
        docs.append(
            Document(
                page_content=page.get_text(),
                metadata={
                    "source": str(pdf_path),
                    "filename": pdf_path.name,
                    "page": page_num + 1,
                    "page_count": len(pdf),
                    "title": pdf_metadata.get("title"),
                    "author": pdf_metadata.get("author"),
                },
            )
        )

D:\Github desktop\RAG\pdf files\pdf (1).pdf
D:\Github desktop\RAG\pdf files\pdf (10).pdf
D:\Github desktop\RAG\pdf files\pdf (11).pdf
D:\Github desktop\RAG\pdf files\pdf (12).pdf
D:\Github desktop\RAG\pdf files\pdf (13).pdf
D:\Github desktop\RAG\pdf files\pdf (14).pdf
D:\Github desktop\RAG\pdf files\pdf (15).pdf
D:\Github desktop\RAG\pdf files\pdf (2).pdf
D:\Github desktop\RAG\pdf files\pdf (3).pdf
D:\Github desktop\RAG\pdf files\pdf (4).pdf
D:\Github desktop\RAG\pdf files\pdf (5).pdf
D:\Github desktop\RAG\pdf files\pdf (6).pdf
D:\Github desktop\RAG\pdf files\pdf (7).pdf
D:\Github desktop\RAG\pdf files\pdf (8).pdf
D:\Github desktop\RAG\pdf files\pdf (9).pdf


# Chunking 

- Add chunk metadata such as the source, page no, chunk index
- This allows easy search of all the chunks that are a part of a specific page of a PDF

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)

In [9]:
len(chunks)

16261

In [10]:
# Assign chunk indices within each document
doc_chunk_counter = {}

for chunk in chunks:

    doc = chunk.metadata["source"]

    if doc not in doc_chunk_counter:
        doc_chunk_counter[doc] = 0

    chunk.metadata["chunk_index"] = doc_chunk_counter[doc]
    doc_chunk_counter[doc] += 1

In [12]:
chunks[300]

Document(metadata={'source': 'D:\\Github desktop\\RAG\\pdf files\\pdf (1).pdf', 'filename': 'pdf (1).pdf', 'page': 51, 'page_count': 432, 'title': '', 'author': '', 'chunk_index': 300}, page_content='Cnaprrn\nArrays\nThe machine can alter the scanned symbol and its behauior\nis in part determined by that symbol, but the symbols on\nthe tape elsewhere do not affect the behauior of the machine.\n- \n" lntelligmt Machinery,"\nA. M. TunrNc, 1948\nThe simplest data structure is the array, which is a contiguous block of memory. It is usually\nused to represent sequences. Given an array A, Ali) denotes the (l + 1)th object stored in the')

# Creating embeddings

In [13]:
model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5",
    device="cuda"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1568.95it/s]


In [14]:
%%time

texts = [c.page_content for c in chunks]
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
)


Batches: 100%|██████████| 509/509 [06:43<00:00,  1.26it/s]

CPU times: total: 6min 44s
Wall time: 6min 44s


# Saving embeddings in FAISS vector store

In [15]:
# import faiss
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

In [16]:
metadata = []

for chunk in chunks:
    metadata.append({
    "text": chunk.page_content,
    "source": chunk.metadata["source"],
    "page": chunk.metadata["page"],
    "chunk_index": chunk.metadata["chunk_index"]
})

In [17]:
chunk_lookup = {}

for chunk in metadata:
    key = (
        chunk["source"],
        chunk["chunk_index"]
    )
    chunk_lookup[key] = chunk
    
from collections import defaultdict
page_lookup = defaultdict(list)

for chunk in metadata:
    key = (
        chunk["source"],
        chunk["page"]
    )
    page_lookup[key].append(chunk)

In [18]:
import pickle

with open("metadata.pkl","wb") as f:
    pickle.dump(metadata,f)

In [19]:
faiss.write_index(index, "docs.index")

In [14]:
# index = faiss.read_index("docs.index")

# Retrieval Block

In [31]:
%%time
# query = "What are some good to have metric properties?"
query = "what are some exercises to strengthen back muscles?"
# query = "who is the CEO of home depot?"

# Questions that fail to retrieve good results
# query = "what is the quarterly performance of walmart?"

query_vector = model.encode(
    [query],
    normalize_embeddings=True,
    convert_to_numpy=True
)
# query_vector
D, I = index.search(query_vector, k=50)


CPU times: total: 1.31 s
Wall time: 1.24 s


# Reranker 

In [32]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-base",
    device="cuda"
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3396.14it/s]


# Weighted ranking 
1. Similarity search - cosine distances
2. Reranking using cross encoder
3. Metadata boost 

In [34]:
%%time
# Retrieve
D, I = index.search(query_vector, k=50)

candidate_chunks = [metadata[i] for i in I[0]]

pairs = [(query, c["text"]) for c in candidate_chunks]

# Reranker scores
rerank_scores = reranker.predict(pairs)

# Normalize FAISS scores
faiss_scores = D[0]
faiss_scores = (faiss_scores - faiss_scores.min()) / (
    faiss_scores.max() - faiss_scores.min() + 1e-8
)

# Normalize reranker scores
rerank_scores = (rerank_scores - rerank_scores.min()) / (
    rerank_scores.max() - rerank_scores.min() + 1e-8
)



CPU times: total: 48.1 s
Wall time: 29.8 s


In [35]:
from collections import Counter
import numpy as np

# Count document occurrences
doc_counts = Counter(
    chunk["source"]
    for chunk in candidate_chunks
)

max_count = max(doc_counts.values())

metadata_scores = []

query_words = set(query.lower().split())

for chunk in candidate_chunks:

    score = 0.0

    # Same-document boost
    score += (
        doc_counts[chunk["source"]]
        / max_count
    )

    # Heading boost (if available)
    if "heading" in chunk:

        overlap = len(
            query_words &
            set(chunk["heading"].lower().split())
        )

        score += overlap / max(1, len(query_words))

    metadata_scores.append(score)

metadata_scores = np.array(metadata_scores)

metadata_scores = (
    metadata_scores
    - metadata_scores.min()
) / (
    metadata_scores.max()
    - metadata_scores.min()
    + 1e-8
)

In [36]:
# Weighted fusion

final = (
      0.6 * faiss_scores
    + 0.25 * rerank_scores
    + 0.15 * metadata_scores
)

# Sort
ranked = sorted(
    zip(final, candidate_chunks),
    key=lambda x: x[0],
    reverse=True,
)

results = [chunk for _, chunk in ranked[:10]]

# Expanding retrieved chunks

Instead of using only a single chunk, we get all the chunks from that page. This avoids missing any relevant context that may not be present in a chunk

In [37]:
expanded_results = []

for chunk in results:

    key = (
        chunk["source"],
        chunk["page"]
    )

    expanded_results.extend(page_lookup[key])

In [38]:
len(expanded_results)

43

In [43]:
context = ""

for r in expanded_results:
    context += f"""
Source: {r['source']}
Page: {r['page']}

{r['text']}

----------------
"""
#context

# Generation

In [28]:
from ollama import chat

In [40]:
prompt = f"""
You are a helpful assistant answering questions from documents.

Rules:
- Answer only from the provided context.
- If the answer is not present, say "I could not find that information."
- Quote important facts when possible.
- Mention the source document if available.

Context:
{context}

Question:
{query}

Answer:
"""

In [41]:
%%time
response = chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response["message"]["content"])

According to the text, the following exercises can help strengthen the back muscles:

1. Fireman's Carry With Partner: This exercise involves lifting a partner and working on functional strength, which develops full-body strength.
2. Side Bends: This exercise strengthens the side of the waist (obliques) and improves flexibility and strength in all directions, including sideways.
3. Grab Ankles Lift: This exercise stretches and strengthens the muscles along the spine, engaging the abdominals as if they were a pair of feet.

Additionally, the text mentions another exercise called "Bridge" which targets the back muscles, specifically:

1. With Arms Folded Across Chest, Heels Flat: This variation involves driving off the legs and pushing off with the hands until weight is placed on the top of the head.
2. Front Bridge (supplementary exercise): This exercise stretches and strengthens other parts of the back, which can be done after completing the Grab Ankles Lift.

Note that these exercises